# DataNexus — Sales Forecasting
*ARIMA + Prophet Ensemble · Time Series Analysis*

In [ ]:
import sys; sys.path.insert(0,'..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from src.pipeline.etl import load_orders, prepare_forecast_series
from src.models.forecasting import run_prophet, monthly_forecast, evaluate_forecast

df = load_orders('../data/orders.csv')
ts = prepare_forecast_series(df)
print(ts.head()); print(ts.shape)

## 1. Time Series Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

ts_indexed = ts.set_index('ds')['y'].resample('W').sum()
result = seasonal_decompose(ts_indexed, model='additive', period=4)

fig = result.plot()
fig.set_size_inches(12, 8)
plt.suptitle('Time Series Decomposition (Weekly)', y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/05_decomposition.png', bbox_inches='tight')
plt.show()

## 2. Stationarity Test (ADF)

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(ts_indexed.dropna())
print(f'ADF Statistic : {adf_result[0]:.4f}')
print(f'p-value       : {adf_result[1]:.4f}')
print('Stationary    :', 'Yes ✓' if adf_result[1] < 0.05 else 'No — differencing required')

## 3. Prophet Forecast with Confidence Intervals

In [ ]:
forecast_df = run_prophet(ts, periods=90)
monthly_fc  = monthly_forecast(forecast_df)

hist_fc = forecast_df[forecast_df['ds'].isin(ts['ds'])].merge(ts, on='ds')
metrics = evaluate_forecast(hist_fc['y'], hist_fc['yhat'])
print(f"MAPE: {metrics['mape']}%   R²: {metrics['r2']}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

actual_monthly = ts.set_index('ds')['y'].resample('ME').sum() / 1000
ax.plot(actual_monthly.index, actual_monthly.values, color='#378ADD', linewidth=2, marker='o', label='Actual')

fwd = monthly_fc[monthly_fc['month_str'].str.contains('2025')]
if len(fwd):
    ax.plot(pd.to_datetime(fwd['month'].astype(str)), fwd['yhat']/1000,
            color='#D85A30', linewidth=2, linestyle='--', marker='D', label='Forecast')
    ax.fill_between(pd.to_datetime(fwd['month'].astype(str)),
                    fwd['yhat_lower']/1000, fwd['yhat_upper']/1000,
                    alpha=0.2, color='#D85A30', label='95% CI')

ax.set_title(f'Sales Forecast · MAPE={metrics["mape"]}% · R²={metrics["r2"]}')
ax.set_ylabel('Revenue ($K)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/06_forecast.png', bbox_inches='tight')
plt.show()